In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
data=pd.read_csv("transaction_dataset.csv")

In [29]:
data.head()

,address,flag,avg_min_between_sent_tnx,avg_min_between_received_tnx,time_diff_between_first_and_last_mins,sent_tnx,received_tnx,number_of_created_contracts,unique_received_from_addresses,unique_sent_to_addresses,...,erc20_max_val_rec,erc20_avg_val_rec,erc20_min_val_sent,erc20_max_val_sent,erc20_avg_val_sent,erc20_uniq_sent_token_name,erc20_uniq_rec_token_name,erc20_most_sent_token_type,erc20_most_rec_token_type,sent_to_contract_flag
0,0x00009277775ac7d0d59eaad8fee3d10ac6c805e8,0,844.26,1093.71,704785.63,721,89,0,40,118,...,1.500000e+07,265586.147600,0.000000,1.683100e+07,271779.920000,39.0,57.0,Cofoundit,Numeraire,0
1,0x0002b44ddb1476db43c868bd494422ee4c136fed,0,12709.07,2958.44,1218216.73,94,8,0,5,14,...,3.650000e+02,57.632615,2.260809,2.260809e+00,2.260809,1.0,7.0,Livepeer Token,Livepeer Token,0
2,0x0002bda54cb772d040f779e88eb453cac0daa244,0,246194.54,2434.02,516729.30,2,10,0,10,2,...,4.428198e+02,65.189009,0.000000,0.000000e+00,0.000000,0.0,8.0,NaN,XENON,0
3,0x00038e6ba2fd5c09aedb96697c8d7b8fa6632e5e,0,10219.60,15785.09,397555.90,25,9,0,7,13,...,1.141223e+04,1555.550174,100.000000,9.029231e+03,3804.076893,1.0,11.0,Raiden,XENON,0
4,0x00062d1dd1afb6fb02540ddad9cdebfe568e0d89,0,36.61,10707.77,382472.42,4598,20,1,7,19,...,9.000000e+04,4934.232147,0.000000,4.500000e+04,13726.659220,6.0,27.0,StatusNetwork,EOS,0


In [43]:
data.info()

<class 'pandas.DataFrame'>
Index: 9816 entries, 0 to 9840
Data columns (total 39 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   address                                              9816 non-null   str    
 1   flag                                                 9816 non-null   int64  
 2   avg_min_between_sent_tnx                             9816 non-null   float64
 3   avg_min_between_received_tnx                         9816 non-null   float64
 4   time_diff_between_first_and_last_mins                9816 non-null   float64
 5   sent_tnx                                             9816 non-null   int64  
 6   received_tnx                                         9816 non-null   int64  
 7   number_of_created_contracts                          9816 non-null   int64  
 8   unique_received_from_addresses                       9816 non-null   int64  
 9   un

## Step 1: Cleaning, standardising, dealing with duplicates 

In [5]:
#drop useless columns
data.drop(['Unnamed: 0', 'Index'], axis=1, inplace=True)

In [6]:
#snake case
data.columns=(
    data.columns
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r"[()]", "", regex=True)
    .str.strip('_')
)

In [7]:
#are there repeating addresses
data["address"].value_counts().max()

np.int64(2)

In [8]:
#checking how many unique vs duplicate
data["address"].value_counts().value_counts()

count
1    9791
2      25
Name: count, dtype: int64

In [9]:
#checking variation in any column across its rows for each address
data.groupby("address").nunique().max(axis=1).value_counts()

1    9809
2       7
Name: count, dtype: int64

In [10]:
#how many different flag values each address has: 1->flag is consistent
data.groupby("address")["flag"].nunique().value_counts()

flag
1    9816
Name: count, dtype: int64

In [11]:
#dropping rows with duplicate addresses
data=data.drop_duplicates(subset="address")

In [12]:
#checking label imbalance
data.groupby('flag').size()

flag
0    7637
1    2179
dtype: int64

## Dealing with constant/near constant features

In [28]:
#identifying features
data.nunique()

address                                                9816
flag                                                      2
avg_min_between_sent_tnx                               5013
avg_min_between_received_tnx                           6223
time_diff_between_first_and_last_mins                  7810
sent_tnx                                                641
received_tnx                                            727
number_of_created_contracts                              20
unique_received_from_addresses                          256
unique_sent_to_addresses                                258
min_value_received                                     4589
max_value_received                                     6302
avg_val_received                                       6767
min_val_sent                                           4719
max_val_sent                                           6647
avg_val_sent                                           5854
total_transactions_including_tnx_to_crea

Near-constant features were not removed automatically. Because rare behaviours may be meaningful in fraud detection, each near-constant feature was assessed by comparing fraud rates across its values. Features with negligible signal were dropped, while features with meaningful fraud separation were transformed into binary indicators representing the presence of rare behaviour.

In [14]:
#dropping features that have zero variance 
data.drop(['erc20_avg_time_between_sent_tnx',
           'erc20_avg_time_between_rec_tnx',
           'erc20_avg_time_between_rec_2_tnx',
           'erc20_avg_time_between_contract_tnx',
           'erc20_min_val_sent_contract',
           'erc20_max_val_sent_contract',
           'erc20_avg_val_sent_contract'],
           axis=1, inplace=True)

In [15]:
#identifying near constant columns
def get_near_constant(data, threshold=0.995):
    near_const = []
    for col in data.columns:
        top_freq = data[col].value_counts(normalize=True, dropna=False).iloc[0]
        if top_freq >= threshold:
            near_const.append(col)
    return near_const

near_const_cols = get_near_constant(data)

In [16]:
near_const_cols

['min_value_sent_to_contract',
 'max_val_sent_to_contract',
 'avg_value_sent_to_contract',
 'total_ether_sent_contracts']

In [17]:
#assessing fraud signal in near-constant columns
# For each near-constant feature, compare fraud rates across its values.
# Signal score = max fraud rate - min fraud rate.
# A higher score means the feature separates fraudulent and non-fraudulent accounts better.

near_const_scores = {}

for col in near_const_cols:
    fraud_rates = data.groupby(col)["flag"].mean()
    signal_score = fraud_rates.max() - fraud_rates.min()
    near_const_scores[col] = signal_score

near_const_scores = pd.Series(near_const_scores).sort_values(ascending=False)

near_const_scores

max_val_sent_to_contract      0.222052
avg_value_sent_to_contract    0.222052
total_ether_sent_contracts    0.222052
min_value_sent_to_contract    0.222030
dtype: float64

In [18]:
drop_near_constant = near_const_scores[near_const_scores < 0.05].index.tolist()
keep_near_constant = near_const_scores[near_const_scores >= 0.05].index.tolist()

In [19]:
keep_near_constant

['max_val_sent_to_contract',
 'avg_value_sent_to_contract',
 'total_ether_sent_contracts',
 'min_value_sent_to_contract']

In [20]:
#all features proved to be useful, but all focus on the interaction with contract. So it was decided to convert to a single binary feature "sent_to_contract_flag"  
data["sent_to_contract_flag"] = (
    data[[
        'max_val_sent_to_contract',
        'avg_value_sent_to_contract',
        'total_ether_sent_contracts',
        'min_value_sent_to_contract'
    ]].sum(axis=1) > 0
).astype(int)

In [21]:
data.drop(['max_val_sent_to_contract',
            'avg_value_sent_to_contract',
            'total_ether_sent_contracts',
            'min_value_sent_to_contract'],
          axis=1, inplace=True)

Sparse count-based features representing ERC20 activity were transformed into binary indicators to capture the presence of token transfer behaviour while avoiding instability caused by extremely low-frequency values.

In [22]:
#one more near constant variable
data['erc20_uniq_sent_addr.1'].value_counts()

erc20_uniq_sent_addr.1
0.0    8959
1.0      26
3.0       1
2.0       1
Name: count, dtype: int64

In [23]:
(data["erc20_uniq_sent_addr"] == data["erc20_uniq_sent_addr.1"]).all()

np.False_

In [27]:
#checking correlation
data[["erc20_uniq_sent_addr", "erc20_uniq_sent_addr.1"]].corr()

KeyError: "['erc20_uniq_sent_addr.1'] not in index"

In [26]:
#no correlation, random column, not in dataset description. decision:drop
data = data.drop(columns=["erc20_uniq_sent_addr.1"])

A duplicated column (erc20_uniq_sent_addr.1) was identified during preprocessing. Correlation analysis showed no relationship with the original feature and the variable was not described in the dataset documentation. It was therefore removed as a data artefact. The original feature, representing the number of unique ERC20 recipient addresses, was retained as a behavioural count variable and further transformed using log scaling and binary indicators.

## Missing values

In [38]:
na_counts = data.isna().sum().sort_values(ascending=False)

In [39]:
na_counts

erc20_most_sent_token_type                             2689
erc20_most_rec_token_type                               871
erc20_uniq_rec_token_name                               829
erc20_uniq_sent_token_name                              829
address                                                   0
erc20_uniq_rec_contract_addr                              0
erc20_total_ether_received                                0
erc20_total_ether_sent                                    0
erc20_total_ether_sent_contract                           0
erc20_uniq_sent_addr                                      0
erc20_uniq_rec_addr                                       0
erc20_avg_val_rec                                         0
erc20_min_val_rec                                         0
erc20_max_val_rec                                         0
flag                                                      0
erc20_min_val_sent                                        0
erc20_max_val_sent                      

Missing values in ERC20-related features were not treated as random but as structural indicators of no ERC20 activity. Therefore, numeric variables were imputed with zero, while categorical variables were assigned a ‘None’ category. This preserves behavioural meaning and avoids distortion introduced by statistical imputation.

In [35]:
#numeric ECR20 varibales
erc20_numeric_cols = [
    'erc20_avg_val_rec',
    'erc20_min_val_rec',
    'erc20_total_ether_received',
    'erc20_total_ether_sent',
    'erc20_total_ether_sent_contract',
    'erc20_uniq_sent_addr',
    'erc20_uniq_rec_addr',
    'erc20_uniq_rec_contract_addr',
    'total_erc20_tnxs',
    'erc20_max_val_rec',
    'erc20_min_val_sent',
    'erc20_max_val_sent',
    'erc20_avg_val_sent',
]

In [36]:
data[erc20_numeric_cols] = data[erc20_numeric_cols].fillna(0)

In [40]:
#categorical ECR20 varibles
data['erc20_most_sent_token_type'] = data['erc20_most_sent_token_type'].fillna("None")
data['erc20_most_rec_token_type'] = data['erc20_most_rec_token_type'].fillna("None")
data['erc20_uniq_sent_token_name'] = data['erc20_uniq_sent_token_name'].fillna("None")
data['erc20_uniq_rec_token_name'] = data['erc20_uniq_rec_token_name'].fillna("None")

In [41]:
#binary feature: erc20 activity flag 
data["has_erc20_activity"] = (data["total_erc20_tnxs"] > 0).astype(int)

In [42]:
data.isna().sum()

address                                                0
flag                                                   0
avg_min_between_sent_tnx                               0
avg_min_between_received_tnx                           0
time_diff_between_first_and_last_mins                  0
sent_tnx                                               0
received_tnx                                           0
number_of_created_contracts                            0
unique_received_from_addresses                         0
unique_sent_to_addresses                               0
min_value_received                                     0
max_value_received                                     0
avg_val_received                                       0
min_val_sent                                           0
max_val_sent                                           0
avg_val_sent                                           0
total_transactions_including_tnx_to_create_contract    0
total_ether_sent               